# Module 6: Annotation Queues

> Self-directed module, ~20 min.

Picking up from Module 5: the online evaluator is scoring new runs as they land. This module closes the loop — once runs have **feedback scores**, route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** from Module 5, this time with `add_to_annotation_queue_id` set instead of an LLM judge. Any run matching the filter is added to the queue automatically. Reviewer labels become the training data for the next iteration of the judge, the agent, or a dataset.

Workflow: create a queue → register a routing rule that listens for failing feedback → generate traces designed to fail → review and label in the UI.

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)


## Step 1 — Create the queue

Queues are named buckets that hold runs awaiting human review. Create one via the LangSmith UI (covered first below) or the SDK — both produce the same result.

### Via the UI

Navigate to **Annotation Queues** in the left sidebar and click **+ Annotation Queue** to open the creation form.

<img src="../images/configure_aq.png" alt="Annotation queue creation form showing the name, description, and dataset fields" style="width: auto; max-height: 360px; border-radius: 8px;">

Fill in three sections:

- **Basic details** — name, description, and an optional default dataset for exporting reviewed runs directly.
- **Annotation rubric** — add reviewer instructions (shown in the sidebar during review) and one or more feedback keys via **+ Add a feedback rubric**. Each key becomes a labeled scoring field reviewers fill in per run.

<img src="../images/aq_rubric.png" alt="Feedback rubric configuration with a scoring key and category descriptions" style="width: auto; max-height: 360px; border-radius: 8px;">

- **Collaborator settings** — reviewer count per run, reservation duration, and optional workspace member assignments.

Click **Create** to finish.

### Via the SDK

`get_or_create_annotation_queue` is idempotent — re-running this notebook reuses the existing queue rather than producing duplicates. Names are workspace-scoped, so a single workspace can hold many queues for different review workflows.

In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="partner-growth-needs-review",
    description="Partner growth runs flagged by the online correctness rule (Module 5) as failing.",
)
print(f"Queue: {queue.name} (id={queue.id})")

## Step 2 — Route flagged runs to the queue

Use the **same `create_run_rule` helper** from Module 5 — this time with `add_to_annotation_queue_id` set instead of an LLM judge. Any run matching the filter is added to the queue automatically.

The filter is what makes it route only the failures:

- `eq(is_root, true)` — only top-level runs (skip child spans)
- `eq(feedback_key, "correctness")` — only runs the online evaluator has scored
- `lt(feedback_score, 0.5)` — only failures

In [ ]:
queue_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name="partner-growth-route-failures",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'lt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID: ", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


Runs won't appear in the queue immediately — the feedback from Module 4 has to land first (~30 seconds after each run completes), and only failing runs get routed. If the queue is empty after Step 3, give it another half-minute.


### Caveat: a well-behaved agent may produce an empty queue

The partner growth agent in this POC is intentionally simple — three tools, one subagent, a clear workflow. On most prompts the online judge will score `correctness=true`, and nothing routes to the queue. That's correct behavior, not a bug, but it does make the queue UI hard to demo.

If the queue ends up empty after Step 3:

- **Raise the threshold:** change `lt(feedback_score, 0.5)` to `lt(feedback_score, 1.0)` and re-create the rule. Routes everything — useful for demos, not for production.
- **Tighten the judge** (in Module 4's rule, editable in the UI): stricter criteria → more failures → more queue entries.
- **Push runs manually:** `client.add_runs_to_annotation_queue(queue_id=queue.id, run_ids=[...])` bypasses the rule entirely. Useful when reviewing specific runs by ID.


## Step 3 — Generate traces that should fail

Both rules are live — the online eval from Module 5 and the queue routing rule from Step 2. Trigger a few more traces and the two-step pipeline kicks in:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.

These prompts are designed to invite the failure modes the judge looks for: unknown clients, unanswerable specifics, future predictions, and one-word answers without justification. Real POCs would tune the judge against actual failure modes observed in production.

In [ ]:
trigger_prompts = [
    "Look up the merchant 'globex-retail' and tell me about their channels.",
    "What was Apex Apparel's exact ROAS last quarter?",
    "How will the apparel category perform next month?",
    "Give me a one-word answer: should I pitch Nestwell Home on affiliate?",
]

for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

## Step 4 — Review in the UI

The deep link below opens the queue's review pane. Each entry shows the run's input, the agent's response, and the full trace tree side-by-side with labeling controls. Reviewer actions include accept, reject, free-text comments, and structured labels (configurable per queue). Keyboard shortcuts speed up bulk review — `j` / `k` to navigate, number keys for scoring.

<img src="../images/annotation_queue.png" alt="Annotation queue review pane with a flagged run open" style="width: auto; max-height: 380px; border-radius: 8px;">


In [ ]:
tenant_id = queue_rule["payload"]["tenant_id"]
queue_url = f"https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}"
print(f"Queue (review UI): {queue_url}")
print(f"Routing rule:      {queue_rule['url']}")


## Common run-rule patterns

The same `create_run_rule` helper handles both online evaluators (with a judge prompt + schema) and queue routing (with `add_to_annotation_queue_id`). Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "web_search")` |

`delete_run_rule(client, rule_id)` tears any of them down.


## Cleanup

Leave the queue and rule active if iterating. To tear them down:

```python
delete_run_rule(client, queue_rule["id"])
client.delete_annotation_queue(queue_id=queue.id)
```


## Recap

| Topic | API |
|---|---|
| Create queue | `get_or_create_annotation_queue(client, name=..., description=...)` |
| Route runs to queue | `create_run_rule(..., filter=..., add_to_annotation_queue_id=queue.id)` |
| Route failures only | filter ends with `lt(feedback_score, 0.5)` |
| Manual push | `client.add_runs_to_annotation_queue(queue_id=..., run_ids=[...])` |

Labeled runs from this queue can be added to the dataset (Module 4) for the next iteration of experiments. Next: `07_deploy_and_govern_optional.ipynb` — ship the agent to LangSmith Deployments (optional).